In [1]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# load processed df
from IPython.utils.capture import capture_output

with capture_output():
    %run ../feature_engineering/feature_creation.ipynb

In [3]:
features_df

,balance__mean__all,balance__median__all,balance__min__all,balance__max__all,balance__std__all,balance__pct_negative__all,balance__pct_below_100__all,balance__pct_below_500__all,n_days__all,balance__mean__30d,...,income__coefficient_of_variation,income__avg_days_between__90d,income__count__90d,paycheck__has_regular,paycheck__consistency_score,paycheck__count__90d,balance__min_before_income__avg__90d,balance__depletion_rate__90d,days_to_deplete_half_balance__avg__90d,DQ_TARGET
0,276.961538,70.09,-1019.10,2732.86,1016.288836,0.475524,0.510490,0.580420,143.0,-497.176071,...,0.672316,14.166667,7.0,0.0,0.686953,5.0,-853.385000,89.173235,9.333333,0.0
1,1674.533585,1758.35,-123.25,3597.09,1159.524803,0.056604,0.094340,0.301887,106.0,2671.932727,...,0.496148,8.400000,11.0,1.0,0.721294,8.0,2507.552222,126.456250,0.000000,0.0
10,-106.435115,-98.40,-1108.49,929.25,501.726601,0.595420,0.633588,0.839695,131.0,-382.525455,...,0.780098,7.000000,13.0,1.0,0.747066,8.0,-643.487000,155.752500,6.000000,0.0
100,-3231.228909,-3752.93,-6273.18,802.40,2080.213280,0.963636,0.963636,0.981818,55.0,-4166.195000,...,0.824226,4.875000,17.0,0.0,0.708656,14.0,-4985.593333,465.608462,0.000000,0.0
1000,1013.427875,615.39,-22.85,12589.57,1545.044777,0.025000,0.112500,0.437500,80.0,601.327692,...,0.393328,5.600000,16.0,0.0,0.859934,13.0,313.198462,400.970455,1.875000,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,49.549474,-44.27,-121.55,776.47,238.842211,0.631579,0.754386,0.929825,57.0,-15.570000,...,0.701242,3.208333,25.0,0.0,0.587806,25.0,-69.263333,142.160000,3.000000,NaN
9996,172.754000,184.21,32.72,297.21,76.111644,0.000000,0.200000,1.000000,30.0,179.411000,...,0.000000,30.500000,3.0,0.0,0.000000,0.0,48.900000,46.321500,3.500000,NaN
9997,993.787302,863.25,96.23,2334.40,558.853248,0.000000,0.015873,0.174603,63.0,939.261304,...,1.320124,2.545455,34.0,0.0,0.777386,13.0,524.793077,309.097551,2.454545,NaN
9998,-897.033333,-777.25,-1929.90,230.30,485.740676,0.941176,0.960784,1.000000,51.0,-908.906250,...,1.086824,5.133333,16.0,0.0,0.000000,0.0,-1118.537273,253.332759,0.000000,NaN


In [4]:
'DQ_TARGET' in features_df.columns

True

# Data Preparation

In [5]:
# keep only labeled rows
df = features_df[features_df["DQ_TARGET"].notna()].copy()

X = df.drop(columns=["DQ_TARGET"])
y = df["DQ_TARGET"].astype(int)

# Forward Selection

In [6]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SequentialFeatureSelector, SelectKBest, f_classif
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

In [7]:
logreg = LogisticRegression(
    penalty="l2",
    solver="liblinear",
    max_iter=2000
)

forward_selector = SequentialFeatureSelector(
    estimator=logreg,
    n_features_to_select=50,
    direction="forward",
    scoring="roc_auc",
    cv=StratifiedKFold(5),
    n_jobs=-1
)

pipeline_forward = Pipeline([
    ("scaler", StandardScaler()),
    ("sfs", forward_selector),
    ("model", logreg)
])

pipeline_forward.fit(X, y)

selected_forward = X.columns[
    pipeline_forward.named_steps["sfs"].get_support()
]

print(f'Length: {len(selected_forward)}')
print(selected_forward)

Length: 50
Index(['balance__pct_negative__all', 'balance__pct_below_100__all',
       'balance__pct_below_500__all', 'n_days__all', 'balance__std__30d',
       'cashflow__net__30d', 'cashflow__mean_daily__30d',
       'cashflow__volatility__30d', 'n_tx__30d', 'balance__std__60d',
       'balance__std__180d', 'debit__total__all', 'tx__max_debit__all',
       'cat_0__cat_net_total__all', 'cat_6__cat_net_total__all',
       'cat_7__cat_net_total__all', 'cat_11__cat_net_total__all',
       'cat_12__cat_net_total__all', 'cat_14__cat_net_total__all',
       'cat_20__cat_net_total__all', 'cat_22__cat_net_total__all',
       'cat_23__cat_net_total__all', 'cat_39__cat_net_total__all',
       'cat_1__cat_n__all', 'cat_3__cat_n__all', 'cat_12__cat_n__all',
       'cat_14__cat_n__all', 'cat_17__cat_n__all', 'cat_22__cat_n__all',
       'cat_23__cat_n__all', 'cat_37__cat_n__all', 'cat_0__cat_net_total__90d',
       'cat_1__cat_net_total__90d', 'cat_6__cat_net_total__90d',
       'cat_7__cat_net_tot

## Backward Selection

In [8]:
cv = StratifiedKFold(n_splits=2, shuffle=True, random_state=42)

logreg = LogisticRegression(
    penalty="l2",
    solver="saga",
    max_iter=1200,
    tol=5e-2,   # looser tol = faster
    n_jobs=-1
    
)

prefilter_k = 80  

backward_selector = SequentialFeatureSelector(
    estimator=logreg,
    n_features_to_select=50,   
    direction="backward",
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1
)

pipeline_backward = Pipeline([
    ("prefilter", SelectKBest(score_func=f_classif, k=prefilter_k)),
    ("scaler", StandardScaler()),
    ("sfs", backward_selector),
    ("model", logreg)
])

pipeline_backward.fit(X, y)

pref_cols = X.columns[pipeline_backward.named_steps["prefilter"].get_support()]
selected_backward = pref_cols[pipeline_backward.named_steps["sfs"].get_support()]

print(f'Length: {len(selected_backward)}')
print(selected_backward)

Length: 50
Index(['balance__std__all', 'balance__pct_negative__all',
       'balance__pct_below_100__all', 'balance__pct_below_500__all',
       'n_days__all', 'n_tx__30d', 'n_days__30d', 'balance__pct_negative__60d',
       'n_days__60d', 'balance__pct_negative__90d', 'n_days__90d',
       'balance__pct_negative__180d', 'n_days__180d', 'tx__n__all',
       'cat_14__cat_net_total__all', 'cat_18__cat_net_total__all',
       'cat_20__cat_net_total__all', 'cat_26__cat_net_total__all',
       'cat_4__cat_n__all', 'cat_12__cat_n__all', 'cat_16__cat_n__all',
       'cat_20__cat_n__all', 'cat_31__cat_n__all', 'cat_35__cat_n__all',
       'cat_6__cat_net_total__90d', 'cat_26__cat_net_total__90d',
       'cat_1__cat_n__90d', 'cat_6__cat_n__90d', 'cat_12__cat_n__90d',
       'cat_18__cat_n__90d', 'cat_19__cat_n__90d', 'cat_23__cat_n__90d',
       'overdraft_fee__count__all', 'overdraft_fee__count__30d',
       'overdraft_fee__total__30d', 'account_fees__count__30d',
       'overdraft_fee__count_

## Model Performance

In [9]:
df = features_df[features_df["DQ_TARGET"].notna()].copy()

y = df["DQ_TARGET"].astype(int)
X = df.drop(columns=["DQ_TARGET"]).apply(pd.to_numeric, errors="coerce").fillna(0)

In [10]:
X_all = X
X_forward = X[selected_forward]
X_backward = X[selected_backward]


cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

logreg = LogisticRegression(
    penalty="l2",
    solver="liblinear",
    max_iter=2000
)

In [11]:
def eval_model(X_subset, y, name):
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("model", logreg)
    ])
    scores = cross_validate(
        pipe,
        X_subset,
        y,
        scoring=["roc_auc", "accuracy"],
        cv=cv,
        n_jobs=-1
    )
    return {
        "Model": name,
        "Num_Features": X_subset.shape[1],
        "Avg ROC-AUC": scores["test_roc_auc"].mean(),
        "Avg Accuracy": scores["test_accuracy"].mean()
    }

In [12]:
results = [
    eval_model(X_all, y, "All Features"),
    eval_model(X_forward, y, "Forward Selected"),
    eval_model(X_backward, y, "Backward Selected"),
]

results_df = pd.DataFrame(results).sort_values("Avg ROC-AUC", ascending=False)
results_df

,Model,Num_Features,Avg ROC-AUC,Avg Accuracy
1,Forward Selected,50,0.767455,0.909082
0,All Features,217,0.759765,0.906368
2,Backward Selected,50,0.756699,0.909470
